<a href="https://colab.research.google.com/github/Fraanas/Big-Data/blob/main/BigData_zajecia9_mllib.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Big Data — zajęcia 9
## MLlib — uczenie maszynowe w ekosystemie Apache Spark

**Tryb pracy:** teoria → pokaz → praca własna studenta  
**Temat przewodni:** jak budować skalowalne pipeline’y uczenia maszynowego w Spark MLlib.

### Cele zajęć
Po zajęciach student:
1. rozumie rolę MLlib w ekosystemie Spark,
2. zna podstawowe pojęcia: `DataFrame`, `Transformer`, `Estimator`, `Pipeline`, `features`, `label`,
3. potrafi przygotować dane do modelu ML w Sparku,
4. umie zakodować zmienne kategoryczne i połączyć cechy w wektor,
5. potrafi trenować modele klasyfikacyjne,
6. umie porównać kilka modeli,
7. zna podstawowe metryki klasyfikacji,
8. potrafi przygotować krótką interpretację biznesową modelu.


## Plan zajęć

1. Wprowadzenie teoretyczne do MLlib.
2. Konfiguracja środowiska.
3. Przygotowanie danych pokazowych.
4. Pipeline MLlib: indeksowanie, kodowanie, wektoryzacja, model.
5. Trenowanie i ewaluacja modelu klasyfikacyjnego.
6. Porównanie modeli.
7. Praca własna studenta — inny scenariusz niż w pokazie.
8. Interpretacja wyników i rekomendacja biznesowa.


# Część 1. Rozbudowane wprowadzenie teoretyczne

## 1. Czym jest MLlib?

MLlib to biblioteka uczenia maszynowego dostępna w Apache Spark.  
Pozwala trenować i stosować modele ML na danych przetwarzanych w środowisku rozproszonym.

MLlib jest szczególnie użyteczny wtedy, gdy:
- dane są zbyt duże, aby wygodnie przetwarzać je w pojedynczym procesie,
- dane są już przetwarzane w Sparku,
- chcemy połączyć ETL, feature engineering i modelowanie w jednym pipeline,
- chcemy wykorzystać Spark SQL / DataFrame API razem z modelami ML.

W praktyce MLlib często pojawia się nie jako „najlepsza biblioteka do każdego modelu”, ale jako element większego pipeline’u danych:
- czyszczenie danych,
- transformacje,
- przygotowanie cech,
- trenowanie modelu,
- ocena,
- predykcja na nowych danych.

## 2. Spark MLlib a klasyczne biblioteki ML

W Pythonie bardzo popularne są biblioteki takie jak `scikit-learn`, `xgboost`, `lightgbm` czy `statsmodels`.

MLlib różni się od nich tym, że:
- pracuje natywnie na Spark DataFrame,
- jest projektowany do pracy rozproszonej,
- dobrze integruje się z dużymi pipeline’ami danych,
- używa wektorowej kolumny `features`,
- modelowanie jest częścią całego procesu Spark.

Ograniczenia MLlib:
- nie ma tak szerokiego ekosystemu modeli jak scikit-learn,
- czasem oferuje mniej opcji tuningu,
- dla małych danych scikit-learn może być prostszy,
- interpretowalność niektórych modeli wymaga dodatkowej pracy.

## 3. DataFrame jako podstawa MLlib

W MLlib pracujemy najczęściej na DataFrame.  
Każdy wiersz to obserwacja, a kolumny zawierają:
- cechy numeryczne,
- cechy kategoryczne,
- etykietę celu,
- ewentualne identyfikatory.

Model MLlib zwykle oczekuje dwóch kolumn:
- `features` — jedna kolumna wektorowa z cechami,
- `label` — zmienna celu.

Przykład:
- `features`: wiek klienta, liczba wizyt, wydatki, segment zakodowany liczbowo,
- `label`: czy klient odejdzie, czy nie.

## 4. Transformer i Estimator

W Spark ML występują dwa bardzo ważne typy obiektów.

### Transformer

Transformer przekształca DataFrame w inny DataFrame.  
Przykłady:
- `OneHotEncoder`,
- `VectorAssembler`,
- wytrenowany model.

Transformer ma metodę:

```python
transform(df)
```

### Estimator

Estimator jest obiektem, który najpierw musi zostać wytrenowany.  
Przykłady:
- `StringIndexer`,
- `LogisticRegression`,
- `RandomForestClassifier`.

Estimator ma metodę:

```python
fit(df)
```

Po wywołaniu `fit()` powstaje model, który jest transformerem.

## 5. Pipeline

Pipeline pozwala połączyć wiele kroków w jeden spójny proces.

Przykład pipeline’u:
1. indeksowanie zmiennych kategorycznych,
2. kodowanie One-Hot,
3. złożenie cech w wektor,
4. trenowanie modelu.

Zalety pipeline’u:
- porządek w kodzie,
- mniejsze ryzyko pominięcia kroku,
- łatwiejsze powtórzenie procesu,
- możliwość wykorzystania tego samego pipeline’u na nowych danych.

## 6. Zmienne kategoryczne

Modele ML nie rozumieją tekstu wprost.  
Jeżeli mamy kolumnę:

```text
segment = basic / silver / gold / vip
```

musimy ją zakodować.

Typowe kroki w Sparku:
1. `StringIndexer` — zamienia kategorie tekstowe na indeksy liczbowe,
2. `OneHotEncoder` — zamienia indeksy na reprezentację wektorową.

To jest ważne, ponieważ proste przypisanie `basic=0`, `silver=1`, `gold=2` mogłoby fałszywie zasugerować modelowi porządek między kategoriami.

## 7. VectorAssembler

`VectorAssembler` łączy wiele kolumn cech w jedną kolumnę `features`.

Przykład:

```python
VectorAssembler(
    inputCols=["age", "income", "visits", "segment_ohe"],
    outputCol="features"
)
```

To jeden z najważniejszych etapów w MLlib.

## 8. Podział na train/test

Model powinien być oceniany na danych, których nie widział podczas trenowania.

Typowy podział:
- 70–80% danych treningowych,
- 20–30% danych testowych.

W Sparku:

```python
train, test = df.randomSplit([0.8, 0.2], seed=42)
```

## 9. Data leakage

Data leakage to sytuacja, w której do modelu trafia informacja, której nie powinien znać w momencie predykcji.

Przykład:
- przewidujemy churn klienta,
- ale jako cechę dajemy informację „data zamknięcia konta”.

Model będzie wyglądał świetnie, ale w praktyce będzie bezużyteczny.

W projektach Big Data leakage jest szczególnie groźny, bo:
- pipeline’y mają wiele źródeł,
- dane są łączone z wielu tabel,
- łatwo przypadkowo dołączyć kolumnę z przyszłości.

## 10. Metryki klasyfikacji

Dla klasyfikacji binarnej często używa się:
- Accuracy,
- Precision,
- Recall,
- F1-score,
- AUC.

### Accuracy
Jaki procent predykcji był poprawny.

Problem: przy niezbalansowanych klasach accuracy może być mylące.

### Precision
Spośród przypadków oznaczonych jako pozytywne, ile było rzeczywiście pozytywnych.

### Recall
Spośród rzeczywiście pozytywnych przypadków, ile model wykrył.

### F1-score
Średnia harmoniczna precision i recall.

### AUC
Mierzy zdolność modelu do rozróżniania klas przy różnych progach decyzyjnych.

## 11. Niezbalansowane klasy

W wielu problemach biznesowych klasa pozytywna jest rzadka:
- awaria maszyny,
- fraud,
- churn,
- rezygnacja z usługi,
- niespłacenie kredytu.

Wtedy model może osiągać wysoką accuracy, przewidując prawie zawsze klasę większościową.  
Dlatego trzeba patrzeć na recall, precision, F1 i AUC.

## 12. Interpretacja modelu

W praktyce nie wystarczy podać metryki.  
Trzeba odpowiedzieć:
- co model przewiduje,
- jak dobry jest model,
- jakie są konsekwencje błędów,
- czy lepszy jest model prostszy czy bardziej złożony,
- jakie dane warto dodać,
- czy model można wdrożyć.

## 13. MLlib w pipeline Big Data

Typowy proces:
1. Dane surowe.
2. Czyszczenie i integracja.
3. Feature engineering.
4. Podział train/test.
5. Trenowanie modelu.
6. Ewaluacja.
7. Predykcja.
8. Zapis wyniku.
9. Monitoring modelu.

Model ML jest tylko jednym etapem większego procesu danych.


## Pytania kontrolne

1. Czym różni się `Transformer` od `Estimator`?
2. Dlaczego MLlib używa kolumny `features`?
3. Po co stosujemy `StringIndexer` i `OneHotEncoder`?
4. Dlaczego potrzebny jest podział train/test?
5. Co to jest data leakage?
6. Dlaczego accuracy może być mylące przy niezbalansowanych klasach?
7. Kiedy lepszy jest prosty model, a kiedy bardziej złożony?


# Część 2. Konfiguracja środowiska

> Jeżeli Spark jest już skonfigurowany, można pominąć komórkę instalacyjną.


In [1]:
SPARK_VERSION = "3.5.8"

!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless -qq
!pip -q uninstall -y dataproc-spark-connect || true
!pip -q install pyspark==3.5.8


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Failed to fetch https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu/dists/jammy/InRelease  Could not connect to ppa.launchpadcontent.net:443 (185.125.190.80), connection timed out
W: Failed to fetch https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu/dists/jammy/InRelease  Unable to connect to ppa.launchpadcontent.net:443:
W: Some index files failed to download. They have been ignored, or old ones used instead.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.8/317.8 MB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, DecisionTreeClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from time import perf_counter
import random

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("BigData_Zajecia_9_MLlib")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark version:", spark.version)
print("Default parallelism:", spark.sparkContext.defaultParallelism)


Spark version: 3.5.8
Default parallelism: 2


# Część 3. Pokaz — klasyfikacja churn klienta

W pokazie budujemy model przewidujący, czy klient może zrezygnować z usługi.

To klasyczny problem biznesowy:
- firmy chcą identyfikować klientów zagrożonych odejściem,
- model może wspierać działania retencyjne,
- ważna jest nie tylko accuracy, ale również recall i precision.


In [3]:
random.seed(909)

rows = []
for i in range(1, 5001):
    tenure = random.randint(1, 60)
    monthly_spend = round(random.uniform(20, 500), 2)
    tickets = random.randint(0, 15)
    app_logins = random.randint(0, 80)
    channel = random.choice(["mobile", "web", "store"])
    segment = random.choice(["basic", "silver", "gold", "vip"])

    # Logika syntetyczna: większe ryzyko churn przy krótkim stażu,
    # niskim wydatku, wielu zgłoszeniach i małej aktywności.
    risk_score = 0
    risk_score += 1 if tenure < 8 else 0
    risk_score += 1 if monthly_spend < 120 else 0
    risk_score += 1 if tickets > 6 else 0
    risk_score += 1 if app_logins < 10 else 0
    risk_score += 1 if segment == "basic" else 0

    churn = 1 if risk_score >= 3 or (tickets > 10 and app_logins < 20) else 0

    rows.append((i, tenure, monthly_spend, tickets, app_logins, channel, segment, churn))

churn_df = spark.createDataFrame(
    rows,
    ["customer_id", "tenure_months", "monthly_spend", "tickets_last_90d", "app_logins_30d", "channel", "segment", "label"]
)

churn_df.show(10, truncate=False)
churn_df.groupBy("label").count().show()


+-----------+-------------+-------------+----------------+--------------+-------+-------+-----+
|customer_id|tenure_months|monthly_spend|tickets_last_90d|app_logins_30d|channel|segment|label|
+-----------+-------------+-------------+----------------+--------------+-------+-------+-----+
|1          |14           |338.71       |6               |30            |mobile |gold   |0    |
|2          |25           |343.31       |13              |65            |web    |silver |0    |
|3          |30           |317.59       |2               |78            |store  |vip    |0    |
|4          |55           |132.49       |4               |67            |store  |basic  |0    |
|5          |5            |473.67       |11              |80            |mobile |basic  |1    |
|6          |58           |358.24       |6               |14            |store  |silver |0    |
|7          |9            |40.73        |15              |33            |web    |basic  |1    |
|8          |15           |268.08       

## Przygotowanie pipeline’u MLlib

W tym przykładzie:
- zmienne kategoryczne: `channel`, `segment`,
- zmienne numeryczne: `tenure_months`, `monthly_spend`, `tickets_last_90d`, `app_logins_30d`,
- etykieta: `label`.

Pipeline:
1. `StringIndexer`,
2. `OneHotEncoder`,
3. `VectorAssembler`,
4. `LogisticRegression`.


In [4]:
categorical_cols = ["channel", "segment"]
numeric_cols = ["tenure_months", "monthly_spend", "tickets_last_90d", "app_logins_30d"]

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_cols
]

encoder = OneHotEncoder(
    inputCols=[f"{c}_idx" for c in categorical_cols],
    outputCols=[f"{c}_ohe" for c in categorical_cols]
)

assembler = VectorAssembler(
    inputCols=numeric_cols + [f"{c}_ohe" for c in categorical_cols],
    outputCol="features"
)

lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=30)

pipeline_lr = Pipeline(stages=indexers + [encoder, assembler, lr])

train, test = churn_df.randomSplit([0.8, 0.2], seed=42)

model_lr = pipeline_lr.fit(train)
pred_lr = model_lr.transform(test)

pred_lr.select("customer_id", "label", "prediction", "probability").show(10, truncate=False)


+-----------+-----+----------+------------------------------------------+
|customer_id|label|prediction|probability                               |
+-----------+-----+----------+------------------------------------------+
|3          |0    |0.0       |[0.9999613483208494,3.8651679150625284E-5]|
|7          |1    |1.0       |[0.018351838832357515,0.9816481611676425] |
|9          |0    |1.0       |[0.3767709610664476,0.6232290389335524]   |
|14         |0    |0.0       |[0.9680628418915808,0.03193715810841924]  |
|20         |0    |0.0       |[0.9944620988586708,0.005537901141329238] |
|24         |0    |0.0       |[0.9827540556655485,0.01724594433445148]  |
|30         |0    |0.0       |[0.9833006138188364,0.016699386181163622] |
|36         |0    |1.0       |[0.33440249660748256,0.6655975033925174]  |
|46         |0    |0.0       |[0.9936769110767285,0.0063230889232714915]|
|47         |1    |0.0       |[0.6828002941194642,0.3171997058805358]   |
+-----------+-----+----------+--------

## Ewaluacja modelu

Dla klasyfikacji binarnej używamy kilku metryk.  
Jedna metryka nie wystarczy do dobrej oceny modelu.


In [5]:
auc_eval = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
acc_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
f1_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
precision_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
recall_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")

print("AUC:", round(auc_eval.evaluate(pred_lr), 4))
print("Accuracy:", round(acc_eval.evaluate(pred_lr), 4))
print("F1:", round(f1_eval.evaluate(pred_lr), 4))
print("Precision:", round(precision_eval.evaluate(pred_lr), 4))
print("Recall:", round(recall_eval.evaluate(pred_lr), 4))

# Macierz pomyłek
pred_lr.groupBy("label", "prediction").count().orderBy("label", "prediction").show()


AUC: 0.9436
Accuracy: 0.907
F1: 0.9036
Precision: 0.9018
Recall: 0.907


In [6]:
# Macierz pomyłek
pred_lr.groupBy("label", "prediction").count().orderBy("label", "prediction").show()


+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    0|       0.0|  769|
|    0|       1.0|   33|
|    1|       0.0|   54|
|    1|       1.0|   79|
+-----+----------+-----+



## Porównanie z Random Forest

Drugi model może uchwycić nieliniowe zależności.  
W praktyce porównujemy prostszy model z bardziej elastycznym modelem i sprawdzamy, czy wzrost złożoności daje realną wartość.


In [7]:
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=40,
    maxDepth=6,
    seed=42
)

pipeline_rf = Pipeline(stages=indexers + [encoder, assembler, rf])

model_rf = pipeline_rf.fit(train)
pred_rf = model_rf.transform(test)

print("RF AUC:", round(auc_eval.evaluate(pred_rf), 4))
print("RF Accuracy:", round(acc_eval.evaluate(pred_rf), 4))
print("RF F1:", round(f1_eval.evaluate(pred_rf), 4))
pred_rf.groupBy("label", "prediction").count().orderBy("label", "prediction").show()


RF AUC: 0.9999
RF Accuracy: 0.9925
RF F1: 0.9925
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    0|       0.0|  801|
|    0|       1.0|    1|
|    1|       0.0|    6|
|    1|       1.0|  127|
+-----+----------+-----+



## Interpretacja części pokazowej

W części pokazowej najważniejsze jest zrozumienie procesu:
1. dane wejściowe,
2. przygotowanie cech,
3. pipeline,
4. trenowanie,
5. predykcja,
6. ewaluacja,
7. porównanie modeli.

Nie chodzi wyłącznie o najwyższą metrykę. W projektach ML ważne są także:
- koszt błędów,
- interpretowalność,
- stabilność,
- możliwość wdrożenia,
- jakość danych.


# Część 4. Praca własna studenta

## Zadanie główne — predykcyjne utrzymanie ruchu maszyn produkcyjnych

W części pokazowej analizowany był churn klienta.  
W pracy własnej student otrzymuje **inny scenariusz**: przewidywanie awarii maszyn.

### Kontekst biznesowy

Firma produkcyjna monitoruje pracę maszyn.  
Dla każdej maszyny zbierane są:
- temperatura,
- drgania,
- ciśnienie,
- liczba godzin pracy od ostatniego serwisu,
- liczba drobnych błędów,
- typ maszyny,
- linia produkcyjna,
- status ostatniego przeglądu.

Celem jest przewidzenie, czy maszyna jest zagrożona awarią w najbliższym okresie.

### Dlaczego to ważne?
Awaria maszyny może oznaczać:
- przestój produkcji,
- koszty naprawy,
- opóźnienia zamówień,
- ryzyko pogorszenia jakości produktu.

W takim problemie szczególnie ważny jest **recall**, bo niewykrycie awarii może być kosztowne.


## Struktura zadania

Student ma wykonać:

1. eksplorację danych,
2. przygotowanie cech,
3. podział train/test,
4. zbudowanie modelu bazowego,
5. zbudowanie drugiego modelu porównawczego,
6. ewaluację metrykami,
7. analizę macierzy pomyłek,
8. krótką rekomendację biznesową.

Zadanie jest celowo inne niż pokaz:
- pokaz dotyczył klientów i churn,
- zadanie dotyczy maszyn i ryzyka awarii,
- inne są cechy, inna interpretacja błędów i inne konsekwencje biznesowe.


In [8]:
# Przygotowanie danych do zadania studenckiego
# Dane są syntetyczne, ale odzwierciedlają problem predictive maintenance.

random.seed(2027)

machine_types = ["press", "cutter", "conveyor", "robot", "packer"]
production_lines = ["line_A", "line_B", "line_C", "line_D"]
maintenance_statuses = ["ok", "delayed", "critical"]

rows = []

for machine_id in range(1, 7001):
    machine_type = random.choice(machine_types)
    line = random.choice(production_lines)
    maintenance_status = random.choices(
        maintenance_statuses,
        weights=[70, 22, 8]
    )[0]

    temperature = round(random.uniform(35, 110), 2)
    vibration = round(random.uniform(0.1, 9.5), 2)
    pressure = round(random.uniform(40, 160), 2)
    hours_since_service = random.randint(10, 2200)
    minor_errors_30d = random.randint(0, 25)
    load_pct = round(random.uniform(30, 100), 2)

    # syntetyczna logika ryzyka
    risk = 0
    risk += 1 if temperature > 85 else 0
    risk += 1 if vibration > 6.5 else 0
    risk += 1 if pressure > 135 or pressure < 55 else 0
    risk += 1 if hours_since_service > 1400 else 0
    risk += 1 if minor_errors_30d > 12 else 0
    risk += 1 if load_pct > 88 else 0
    risk += 1 if maintenance_status == "critical" else 0
    risk += 1 if maintenance_status == "delayed" and hours_since_service > 1100 else 0

    # Klasa pozytywna nie jest bardzo częsta
    failure_risk = 1 if risk >= 4 or (temperature > 95 and vibration > 7.0) else 0

    rows.append((
        machine_id,
        machine_type,
        line,
        maintenance_status,
        temperature,
        vibration,
        pressure,
        hours_since_service,
        minor_errors_30d,
        load_pct,
        failure_risk
    ))

maintenance_df = spark.createDataFrame(
    rows,
    [
        "machine_id",
        "machine_type",
        "production_line",
        "maintenance_status",
        "temperature",
        "vibration",
        "pressure",
        "hours_since_service",
        "minor_errors_30d",
        "load_pct",
        "label"
    ]
)

maintenance_df.show(10, truncate=False)
maintenance_df.groupBy("label").count().show()


+----------+------------+---------------+------------------+-----------+---------+--------+-------------------+----------------+--------+-----+
|machine_id|machine_type|production_line|maintenance_status|temperature|vibration|pressure|hours_since_service|minor_errors_30d|load_pct|label|
+----------+------------+---------------+------------------+-----------+---------+--------+-------------------+----------------+--------+-----+
|1         |press       |line_D         |ok                |84.14      |7.55     |115.09  |761                |12              |42.6    |0    |
|2         |packer      |line_B         |ok                |86.51      |0.82     |119.23  |1247               |2               |47.83   |0    |
|3         |conveyor    |line_D         |ok                |46.74      |8.36     |141.09  |908                |12              |61.77   |0    |
|4         |cutter      |line_D         |ok                |90.25      |3.06     |141.11  |794                |24              |34.33   

## Etap 1 — eksploracja danych

### Wykonaj

1. Sprawdź schemat danych.
2. Policz liczbę rekordów.
3. Sprawdź rozkład klasy `label`.
4. Sprawdź średnie wartości cech numerycznych dla klas `0` i `1`.
5. Sprawdź liczbę maszyn według `machine_type`, `production_line` i `maintenance_status`.

### Pytania interpretacyjne

- Czy klasy są zbalansowane?
- Które cechy mogą być powiązane z ryzykiem awarii?
- Czy accuracy będzie wystarczającą metryką?


In [22]:
# TODO 1A: sprawdź schemat i liczbę rekordów
maintenance_df.printSchema()
print("Rows:", maintenance_df.count())


root
 |-- machine_id: long (nullable = true)
 |-- machine_type: string (nullable = true)
 |-- production_line: string (nullable = true)
 |-- maintenance_status: string (nullable = true)
 |-- temperature: double (nullable = true)
 |-- vibration: double (nullable = true)
 |-- pressure: double (nullable = true)
 |-- hours_since_service: long (nullable = true)
 |-- minor_errors_30d: long (nullable = true)
 |-- load_pct: double (nullable = true)
 |-- label: long (nullable = true)

Rows: 7000


In [23]:
# TODO 1B: sprawdź rozkład klasy label
maintenance_df.groupBy("label").count().show()


+-----+-----+
|label|count|
+-----+-----+
|    0| 5808|
|    1| 1192|
+-----+-----+



In [25]:
# TODO 1C: porównaj średnie wartości cech numerycznych dla label=0 i label=1
# Wskazówka:
maintenance_df.groupBy("label").agg(
     F.round(F.avg("temperature"), 2).alias("avg_temperature"),
     F.round(F.avg("vibration"), 2).alias("avg_vibration"),
     F.round(F.avg("hours_since_service"), 2).alias("avg_hours_since_service"),
     F.round(F.avg("minor_errors_30d"), 2).alias("avg_minor_errors")
).show()


+-----+---------------+-------------+-----------------------+----------------+
|label|avg_temperature|avg_vibration|avg_hours_since_service|avg_minor_errors|
+-----+---------------+-------------+-----------------------+----------------+
|    0|          69.35|         4.45|                 1024.9|           11.88|
|    1|          85.51|         6.48|                1434.33|           15.04|
+-----+---------------+-------------+-----------------------+----------------+



In [26]:
# TODO 1D: rozkład zmiennych kategorycznych
maintenance_df.groupBy("machine_type").count().show()
maintenance_df.groupBy("production_line").count().show()
maintenance_df.groupBy("maintenance_status").count().show()


+------------+-----+
|machine_type|count|
+------------+-----+
|       press| 1429|
|      packer| 1389|
|       robot| 1369|
|    conveyor| 1372|
|      cutter| 1441|
+------------+-----+

+---------------+-----+
|production_line|count|
+---------------+-----+
|         line_D| 1743|
|         line_B| 1729|
|         line_A| 1775|
|         line_C| 1753|
+---------------+-----+

+------------------+-----+
|maintenance_status|count|
+------------------+-----+
|          critical|  553|
|                ok| 4934|
|           delayed| 1513|
+------------------+-----+



## Etap 2 — przygotowanie pipeline’u

### Wymagania

Utwórz pipeline MLlib, który zawiera:
1. `StringIndexer` dla zmiennych kategorycznych,
2. `OneHotEncoder`,
3. `VectorAssembler`,
4. model klasyfikacyjny.

### Zmienne kategoryczne

- `machine_type`,
- `production_line`,
- `maintenance_status`.

### Zmienne numeryczne

- `temperature`,
- `vibration`,
- `pressure`,
- `hours_since_service`,
- `minor_errors_30d`,
- `load_pct`.

### Etykieta

- `label`.


In [28]:
# TODO 2A: zdefiniuj kolumny cech

categorical_cols_task = [
   'machine_type', 'production_line', 'maintenance_status'
]

numeric_cols_task = [
    'temperature', 'vibration', 'pressure', 'hours_since_service', 'minor_errors_30d', 'load_pct'
]


In [29]:
maintenance_df.show(15)

+----------+------------+---------------+------------------+-----------+---------+--------+-------------------+----------------+--------+-----+
|machine_id|machine_type|production_line|maintenance_status|temperature|vibration|pressure|hours_since_service|minor_errors_30d|load_pct|label|
+----------+------------+---------------+------------------+-----------+---------+--------+-------------------+----------------+--------+-----+
|         1|       press|         line_D|                ok|      84.14|     7.55|  115.09|                761|              12|    42.6|    0|
|         2|      packer|         line_B|                ok|      86.51|     0.82|  119.23|               1247|               2|   47.83|    0|
|         3|    conveyor|         line_D|                ok|      46.74|     8.36|  141.09|                908|              12|   61.77|    0|
|         4|      cutter|         line_D|                ok|      90.25|     3.06|  141.11|                794|              24|   34.33

In [33]:
# TODO 2B: przygotuj indexery, encoder i assembler
# Wskazówka: użyj podobnej struktury jak w części pokazowej.

from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_cols_task
]

encoder = OneHotEncoder(
    inputCols=[f"{c}_idx" for c in categorical_cols_task],
    outputCols=[f"{c}_ohe" for c in categorical_cols_task]
)

assembler = VectorAssembler(
    inputCols=numeric_cols_task + [f"{c}_ohe" for c in categorical_cols_task],
    outputCol="features"
)

+----------+-----+----------+------------------------------------------+
|machine_id|label|prediction|probability                               |
+----------+-----+----------+------------------------------------------+
|3         |0    |0.0       |[0.9684222082225886,0.03157779177741138]  |
|7         |0    |0.0       |[0.9268830714998151,0.07311692850018492]  |
|9         |0    |0.0       |[0.9320999791335154,0.06790002086648461]  |
|14        |0    |0.0       |[0.9997970061656037,2.0299383439625895E-4]|
|20        |0    |0.0       |[0.563355370946954,0.436644629053046]     |
|24        |0    |0.0       |[0.9996532666415913,3.467333584087351E-4] |
|30        |0    |0.0       |[0.902288096737545,0.09771190326245505]   |
|36        |0    |0.0       |[0.9823278929662608,0.017672107033739226] |
|46        |0    |0.0       |[0.9988957331046021,0.0011042668953978874]|
|47        |0    |0.0       |[0.9296262226435363,0.07037377735646366]  |
+----------+-----+----------+----------------------

## Etap 3 — podział danych

Podziel dane na zbiór treningowy i testowy.

### Wymagania
- 80% train,
- 20% test,
- użyj `seed=42`,
- sprawdź liczebności obu zbiorów.


In [34]:
# TODO 3A: wykonaj podział train/test
train, test = maintenance_df.randomSplit([0.8, 0.2], seed=42)


## Etap 4 — model bazowy: Logistic Regression

Zbuduj model bazowy z użyciem `LogisticRegression`.

### Wymagania
1. Dodaj model do pipeline’u.
2. Wytrenuj pipeline na zbiorze treningowym.
3. Wykonaj predykcję na zbiorze testowym.
4. Pokaż kolumny:
   - `machine_id`,
   - `label`,
   - `prediction`,
   - `probability`.

### Dlaczego Logistic Regression?
To dobry model bazowy:
- prosty,
- szybki,
- relatywnie interpretowalny,
- dobry punkt odniesienia dla bardziej złożonych modeli.


In [35]:
# TODO 4A: zbuduj i wytrenuj model Logistic Regression
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=30)


In [36]:
pipeline_lr = Pipeline(stages=indexers + [encoder, assembler, lr])

train, test = maintenance_df.randomSplit([0.8, 0.2], seed=42)

model_lr = pipeline_lr.fit(train)
pred_lr = model_lr.transform(test)

pred_lr.select("machine_id", "label", "prediction", "probability").show(10, truncate=False)

+----------+-----+----------+------------------------------------------+
|machine_id|label|prediction|probability                               |
+----------+-----+----------+------------------------------------------+
|3         |0    |0.0       |[0.9684222082225886,0.03157779177741138]  |
|7         |0    |0.0       |[0.9268830714998151,0.07311692850018492]  |
|9         |0    |0.0       |[0.9320999791335154,0.06790002086648461]  |
|14        |0    |0.0       |[0.9997970061656037,2.0299383439625895E-4]|
|20        |0    |0.0       |[0.563355370946954,0.436644629053046]     |
|24        |0    |0.0       |[0.9996532666415913,3.467333584087351E-4] |
|30        |0    |0.0       |[0.902288096737545,0.09771190326245505]   |
|36        |0    |0.0       |[0.9823278929662608,0.017672107033739226] |
|46        |0    |0.0       |[0.9988957331046021,0.0011042668953978874]|
|47        |0    |0.0       |[0.9296262226435363,0.07037377735646366]  |
+----------+-----+----------+----------------------

## Etap 5 — ewaluacja modelu bazowego

### Wymagania

Policz:
- AUC,
- accuracy,
- F1,
- precision,
- recall,
- macierz pomyłek.

### Interpretacja

W problemie awarii maszyn szczególnie ważne jest pytanie:

> Czy bardziej boimy się fałszywego alarmu, czy niewykrytej awarii?

Niewykryta awaria może być bardzo kosztowna, dlatego recall dla klasy awarii może być ważniejszy niż sama accuracy.


In [37]:
# TODO 5A: ewaluacja modelu bazowego

auc_eval = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
acc_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
f1_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
precision_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
recall_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")

print("AUC:", round(auc_eval.evaluate(pred_lr), 4))
print("Accuracy:", round(acc_eval.evaluate(pred_lr), 4))
print("F1:", round(f1_eval.evaluate(pred_lr), 4))
print("Precision:", round(precision_eval.evaluate(pred_lr), 4))
print("Recall:", round(recall_eval.evaluate(pred_lr), 4))



AUC: 0.9322
Accuracy: 0.8871
F1: 0.8791
Precision: 0.8795
Recall: 0.8871


In [38]:
# TODO 5B: macierz pomyłek
pred_lr.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    0|       0.0| 1059|
|    0|       1.0|   40|
|    1|       0.0|  111|
|    1|       1.0|  128|
+-----+----------+-----+



## Etap 6 — model porównawczy: Random Forest

Zbuduj drugi model, używając `RandomForestClassifier`.

### Wymagania
1. Użyj tego samego przygotowania cech.
2. Wytrenuj model Random Forest.
3. Policz te same metryki co dla Logistic Regression.
4. Porównaj wyniki.
5. Sprawdź, czy bardziej złożony model daje zauważalną korzyść.

### Pytanie
Czy wzrost złożoności modelu jest uzasadniony biznesowo?


In [47]:
from xgboost.spark import SparkXGBRegressor, SparkXGBClassifier

In [52]:
xgb = SparkXGBClassifier(
    features_col="features",
    label_col="label",
    maxDepth=6,
    seed=42
)
pipeline_xgb = Pipeline(stages=indexers + [encoder, assembler, xgb])

model_xgb = pipeline_xgb.fit(train)
pred_xgb = model_xgb.transform(test)

# TODO 6B: ewaluacja Random Forest

print("RF AUC:", round(auc_eval.evaluate(pred_xgb), 4))
print("RF Accuracy:", round(acc_eval.evaluate(pred_xgb), 4))
print("RF F1:", round(f1_eval.evaluate(pred_xgb), 4))
print("Precision:", round(precision_eval.evaluate(pred_xgb), 4))
print("Recall:", round(recall_eval.evaluate(pred_xgb), 4))
pred_xgb.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

INFO:XGBoost-PySpark:Running xgboost-3.2.0 on 1 workers with
	booster params: {'objective': 'binary:logistic', 'device': 'cpu', 'maxDepth': 6, 'seed': 42, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 100}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
INFO:XGBoost-PySpark:Finished xgboost training!


RF AUC: 0.9996
RF Accuracy: 0.9918
RF F1: 0.9917
Precision: 0.9918
Recall: 0.9918
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    0|       0.0| 1097|
|    0|       1.0|    2|
|    1|       0.0|    9|
|    1|       1.0|  230|
+-----+----------+-----+



In [40]:
# TODO 6A: zbuduj i wytrenuj Random Forest
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=40,
    maxDepth=6,
    seed=42
)

pipeline_rf = Pipeline(stages=indexers + [encoder, assembler, rf])

model_rf = pipeline_rf.fit(train)
pred_rf = model_rf.transform(test)

In [42]:
# TODO 6B: ewaluacja Random Forest

print("RF AUC:", round(auc_eval.evaluate(pred_rf), 4))
print("RF Accuracy:", round(acc_eval.evaluate(pred_rf), 4))
print("RF F1:", round(f1_eval.evaluate(pred_rf), 4))
print("Precision:", round(precision_eval.evaluate(pred_rf), 4))
print("Recall:", round(recall_eval.evaluate(pred_rf), 4))
pred_rf.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

RF AUC: 0.9802
RF Accuracy: 0.9312
RF F1: 0.9248
Precision: 0.9354
Recall: 0.9312
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    0|       0.0| 1097|
|    0|       1.0|    2|
|    1|       0.0|   90|
|    1|       1.0|  149|
+-----+----------+-----+



## Etap 7 — interpretacja i rekomendacja

Przygotuj krótką rekomendację dla kierownika utrzymania ruchu.

### Odpowiedz

1. Który model wybrałbyś do dalszych testów?
2. Która metryka jest najważniejsza i dlaczego?
3. Czy model ma tendencję do fałszywych alarmów?
4. Czy model może pomijać ryzykowne maszyny?
5. Jakie dodatkowe dane warto byłoby zbierać?
6. Jak można wykorzystać predykcję w praktyce?
7. Czy model powinien działać automatycznie, czy jako wsparcie decyzji człowieka?

### Oczekiwana długość
Napisz 8–12 zdań.


In [44]:
print('Logistic regression')
print("AUC:", round(auc_eval.evaluate(pred_lr), 4))
print("Accuracy:", round(acc_eval.evaluate(pred_lr), 4))
print("F1:", round(f1_eval.evaluate(pred_lr), 4))
print("Precision:", round(precision_eval.evaluate(pred_lr), 4))
print("Recall:", round(recall_eval.evaluate(pred_lr), 4))
print('')
print('Random Forest')
print("RF AUC:", round(auc_eval.evaluate(pred_rf), 4))
print("RF Accuracy:", round(acc_eval.evaluate(pred_rf), 4))
print("RF F1:", round(f1_eval.evaluate(pred_rf), 4))
print("Precision:", round(precision_eval.evaluate(pred_rf), 4))
print("Recall:", round(recall_eval.evaluate(pred_rf), 4))

Logistic regression
AUC: 0.9322
Accuracy: 0.8871
F1: 0.8791
Precision: 0.8795
Recall: 0.8871

Random Forest
RF AUC: 0.9802
RF Accuracy: 0.9312
RF F1: 0.9248
Precision: 0.9354
Recall: 0.9312


In [45]:
print('Logistic regression')
pred_lr.groupBy("label", "prediction").count().orderBy("label", "prediction").show()
print('')
print('Random Forest')
pred_rf.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

Logistic regression
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    0|       0.0| 1059|
|    0|       1.0|   40|
|    1|       0.0|  111|
|    1|       1.0|  128|
+-----+----------+-----+


Random Forest
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    0|       0.0| 1097|
|    0|       1.0|    2|
|    1|       0.0|   90|
|    1|       1.0|  149|
+-----+----------+-----+



In [21]:
# TODO 7A: rekomendacja końcowa
# Wpisz odpowiedź jako komentarz:
#
# 1. Wybrany model:
# 2. Najważniejsza metryka:
# 3. Główne ryzyko błędu:
# 4. Rekomendowane użycie:
# 5. Dane do uzupełnienia:


## Praca domowa / rozwinięcie

Wybierz jedną ścieżkę:

1. **Cross-validation**  
   Dodaj `CrossValidator` i porównaj kilka wartości `maxDepth` albo `regParam`.

2. **Balans klas**  
   Zbadaj, jak zmienia się model, gdy klasa awarii jest rzadsza.

3. **Nowe cechy**  
   Dodaj cechę syntetyczną, np. `risk_index = temperature * vibration`.

4. **Interpretacja błędów**  
   Wyświetl przypadki, w których model przewidział brak awarii, a rzeczywista etykieta to awaria.

5. **Wdrożenie**  
   Opisz w 1 stronie, jak taki model mógłby działać w systemie produkcyjnym.
